## Step 1. Load SciQ dataset
1, Load the SciQ dataset

2, Print the dataset structure

3, Print one sample from the training set

In [ ]:
import sys
!{sys.executable} -m pip install --no-cache-dir datasets

  Using cached datasets-4.6.1-py3-none-any.whl.metadata (19 kB)
  Using cached filelock-3.25.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached numpy-2.4.2-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached pyarrow-23.0.1-cp313-cp313-win_amd64.whl.metadata (3.1 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-3.0.1-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached xxhash-3.6.0-cp313-cp313-win_amd64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.18-py313-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-1.5.0-py3-none-any.whl.metadata (13 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached aiohttp-3.13.3-cp313-cp313-win_amd64.

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To 

In [6]:
from datasets import load_dataset

# Download the SciQ dataset
dataset = load_dataset("sciq")

# Print the dataset structure
print("Dataset structure:\n", dataset)

# Print one sample from the training set
print("Sample from training set:\n", dataset["train"][0])

c:\Users\S\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\S\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\S\.cache\huggingface\hub\datasets--sciq. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to act

Dataset structure:
 DatasetDict({
    train: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 11679
    })
    validation: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 1000
    })
})
Sample from training set:
 {'question': 'What type of organism is commonly used in preparation of foods such as cheese and yogurt?', 'distractor3': 'viruses', 'distractor1': 'protozoa', 'distractor2': 'gymnosperms', 'correct_answer': 'mesophilic organisms', 'support': 'Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many 

Use unique support paragraphs as documents, and store the gold doc id per question.

In [7]:
documents = []
support_to_id = {}

# Build unique corpus
for ex in dataset["train"]:
    s = ex["support"]
    if s not in support_to_id:
        support_to_id[s] = len(documents)
        documents.append({"id": support_to_id[s], "text": s})

# Gold doc id for each training question
gold_doc_id = [support_to_id[ex["support"]] for ex in dataset["train"]]

print("Unique documents:", len(documents))
print("Example doc:", documents[0])
print("Gold doc id for train[0]:", gold_doc_id[0])

Unique documents: 10474
Example doc: {'id': 0, 'text': 'Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.'}
Gold doc id for train[0]: 0


Build embeddings + FAISS index

In [ ]:
import sys
!{sys.executable} -m pip install --no-cache-dir sentence-transformers faiss-cpu

  Using cached sentence_transformers-5.2.3-py3-none-any.whl.metadata (16 kB)
  Using cached faiss_cpu-1.13.2-cp313-cp313-win_amd64.whl.metadata (7.6 kB)
  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.10.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached regex-2026.2.28-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer_slim-0.24.0-py3-none-any.whl.metadata (4.2 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using ca

ERROR: Could not install packages due to an OSError: [WinError 32] 另一个程序正在使用此文件，进程无法访问。: 'c:\\Users\\S\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\sympy\\combinatorics\\tests\\test_polyhedron.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [doc["text"] for doc in documents]
embeddings = model.encode(texts, convert_to_numpy=True)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print("Index size:", index.ntotal)

c:\Users\S\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\S\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1563.39it/s, Materializing 

Index size: 10474


#### Retriever baseline

compute Recall@1 / @3 / @5 for your SciQ retriever (before any LLM)

build corpus (dedup), build FAISS, compute Recall@k

Embedding: all-MiniLM-L6-v2, FAISS IndexFlatL2, no chunking, document-level retrieval.

These are the baseline retrieval metrics

In [13]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1) Load dataset
dataset = load_dataset("sciq")

# 2) Build unique document corpus + mapping (support_text -> doc_id)
documents = []
support_to_id = {}

for ex in dataset["train"]:
    s = ex["support"]
    if s not in support_to_id:
        support_to_id[s] = len(documents)
        documents.append({"id": support_to_id[s], "text": s})

print("Unique documents:", len(documents))

# 3) Embed documents + build FAISS index
model = SentenceTransformer("all-MiniLM-L6-v2")

doc_texts = [d["text"] for d in documents]
doc_emb = model.encode(doc_texts, convert_to_numpy=True, show_progress_bar=True)

index = faiss.IndexFlatL2(doc_emb.shape[1])
index.add(doc_emb)

print("Index size:", index.ntotal)

# 4) Recall@k evaluation
def recall_at_k(split="train", k=3, n=500):
    correct = 0
    # Use a subset n for speed; set n=None to run all
    data = dataset[split] if n is None else dataset[split].select(range(min(n, len(dataset[split]))))

    for ex in data:
        q = ex["question"]
        gold = support_to_id[ex["support"]]  # gold document id

        q_emb = model.encode([q], convert_to_numpy=True)
        _, retrieved = index.search(q_emb, k)  # retrieved shape: (1, k)

        if gold in retrieved[0]:
            correct += 1

    return correct / len(data)

for k in [1, 3, 5]:
    r = recall_at_k(split="train", k=k, n=500)  # change split to "validation" if you prefer
    print(f"Recall@{k}: {r:.4f}")

Unique documents: 10474


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1520.49it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 328/328 [00:33<00:00,  9.89it/s]


Index size: 10474
Recall@1: 0.4520
Recall@3: 0.5960
Recall@5: 0.6520


build MCQ options + correct letter

In [14]:
import random

def make_mcq(example, seed=0):
    rnd = random.Random(seed)

    # SciQ format: three separate distractor fields
    choices = [
        example["distractor1"],
        example["distractor2"],
        example["distractor3"],
        example["correct_answer"],
    ]

    rnd.shuffle(choices)

    correct_idx = choices.index(example["correct_answer"])
    correct_letter = "ABCD"[correct_idx]

    return example["question"], choices, correct_letter


def format_prompt(question, choices):
    return (
        "Answer the multiple-choice question.\n"
        "Reply with ONLY A, B, C, or D.\n\n"
        f"Question: {question}\n"
        f"A) {choices[0]}\n"
        f"B) {choices[1]}\n"
        f"C) {choices[2]}\n"
        f"D) {choices[3]}\n\n"
        "Answer:"
    )


def extract_letter(text):
    text = text.strip().upper()
    for ch in text:
        if ch in ["A", "B", "C", "D"]:
            return ch
    return None

LLM-only baseline with a local HF model (TinyLlama-1.1B-Chat-v1.0)

In [19]:
from datasets import load_dataset
from transformers import pipeline

dataset = load_dataset("sciq")

# Pick a model that can follow "output only A/B/C/D".
# If you have a GPU, this will be much faster.
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device="cpu",   # or just omit device/device_map
    max_length=None
)

def llm_answer_letter(prompt):
    out = pipe(
        prompt,
        max_new_tokens=5,
        do_sample=False,   # deterministic
        temperature=0.0,
        return_full_text=False
    )[0]["generated_text"]
    return extract_letter(out)

def eval_llm_only(split="validation", n=200, seed=0):
    data = dataset[split].select(range(min(n, len(dataset[split]))))

    correct = 0
    total = 0
    bad = 0

    for i, ex in enumerate(data):
        q, choices, gold = make_mcq(ex, seed=seed + i)  # vary seed so shuffling differs per example
        prompt = format_prompt(q, choices)

        pred = llm_answer_letter(prompt)
        total += 1

        if pred is None:
            bad += 1
            continue

        if pred == gold:
            correct += 1

    acc = correct / total
    return acc, bad, total

acc, bad, total = eval_llm_only(split="validation", n=200)
print(f"LLM-only accuracy: {acc:.4f}  (unparsed={bad}/{total})")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1386.47it/s, Materializing param=model.norm.weight]                              
Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM-only accuracy: 0.2050  (unparsed=5/200)


In [21]:
def retrieve_supports(question, k=3):
    q_emb = model.encode([question], convert_to_numpy=True)
    _, idxs = index.search(q_emb, k)  # idxs shape (1, k)
    return [documents[i]["text"] for i in idxs[0]]

def format_rag_prompt(context_passages, question, choices):
    context = "\n\n".join([f"[Context {i+1}]\n{p}" for i, p in enumerate(context_passages)])
    return (
        "Use the context to answer the multiple-choice question.\n"
        "Reply with ONLY A, B, C, or D. No explanation.\n\n"
        f"{context}\n\n"
        f"Question: {question}\n"
        f"A) {choices[0]}\n"
        f"B) {choices[1]}\n"
        f"C) {choices[2]}\n"
        f"D) {choices[3]}\n\n"
        "Answer:"
    )

def eval_rag(split="validation", n=200, seed=0, k=3):
    data = dataset[split].select(range(min(n, len(dataset[split]))))

    correct = 0
    total = 0
    bad = 0

    for i, ex in enumerate(data):
        q, choices, gold = make_mcq(ex, seed=seed + i)

        # retrieve top-k supports using the QUESTION as query
        ctx = retrieve_supports(q, k=k)

        prompt = format_rag_prompt(ctx, q, choices)
        pred = llm_answer_letter(prompt)

        total += 1
        if pred is None:
            bad += 1
            continue
        if pred == gold:
            correct += 1

    return correct / total, bad, total

In [22]:
for k in [1, 3, 5]:
    acc_rag, bad_rag, total = eval_rag("validation", n=200, k=k)
    print(f"RAG accuracy (k={k}): {acc_rag:.4f} (unparsed={bad_rag}/{total})")

RAG accuracy (k=1): 0.2550 (unparsed=5/200)
RAG accuracy (k=3): 0.2550 (unparsed=2/200)
RAG accuracy (k=5): 0.2200 (unparsed=4/200)
